In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True,exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib as plt


# data loading and RUL computation

In [ ]:
from src.data.loading import _validate_dataset , load_cmapss_file , load_rul , load_test , load_train , load_readme , sensor_columns  , operation_columns 


In [14]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from src.utils import load_cmapss , add_rul


## what is cmapss?
- cmapss is a high fidelity turbofan engine simulator developed by NASA for research in aircraft engine health monitoring and prognostics
## what is RUL (Remaining Useful Life)
- RUL is the number of operating cycles left before an engine reaches failure
## what is final cycle evaluation?
- in the test set engines are truncated before failure .
- The model must predict RUL only at the **final observed cycle** of each engine . These predictions are compared against the true hidden RUL values.

### why cap RUL ?
in CMAPSS early cycle of an engine can have very large RUL values.
capping RUL means limiting the maximum target value to a fixed threshold.
### what problem does it solve ?
1. **reduces target imbalances**
without capping , many early - cycle samples dominate the dataset with large RUL values.
2. **Stabilizes Training**
Large target increase variance and make regression harder to optimize
3. **Focuses on actionable regime**
predicting on lower RUL is much more important then capping at Higher RUL eg 180 200+

capping makes the model focus on the degradation region that matters most.


## why engine wise ?
RUL depends on the failure cycle of each **individual engine**.
Since every engine fails at a different time, RUL must be computed seperately per engine trajectory.

## why not row wise ?
row wise calculation ignores engine boundaries and mixes different trajectories.
This would produce incorrect RUL values because rows from different engines do not share the same failure time.

In [15]:
import pandas as pd
import numpy as np

df = load_cmapss("..\\data\\raw\\train_FD001.txt")
df = add_rul(df)

df.head()


,unit,cycle,op_1,op_2,op_3,s_1,s_2,s_3,s_4,s_5,...,s_14,s_15,s_16,s_17,s_18,s_19,s_20,s_21,max_cycle,RUL
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190,192,140
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236,192,140
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442,192,140
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739,192,140
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044,192,140


In [16]:
from sklearn.model_selection import train_test_split
engine_ids = df['unit'].unique()
train_engine_ids , val_engine_ids = train_test_split(engine_ids,random_state=42,test_size=0.2)
train_set_df = df[df['unit'].isin(train_engine_ids)]
val_set_df = df[df['unit'].isin(val_engine_ids)]


## why only s_ features?
the S_features are sensor measurements that directly reflect the engine's physical condition.
They contain the degradation information needed to predict RUL.

## why exclude cycle (for now)?
the cycle number is just a time index not a physical measurement.
including it can let the model rely on time progression instead of learning true degradation patterns.
we exclude it to force the model to learn from sensor behaviour.


In [17]:
features = [c for c in df.columns if c.startswith("s_")]

X_train = train_set_df[features]
y_train = train_set_df["RUL"]

X_val = val_set_df[features]
y_val = val_set_df["RUL"]


## why fit only on train?
scaling parameters must be only computed from the training data so that the model learns using information available  during training.
## what leakage does this prevent?
if we fit the scaler on validation set or test data, we use information from unseen data to compute scaling statistics.
this leaks future information into training and leads to overly optimistic evaluation result.

In [18]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)